# Playground Series S6E7 — Predicting Student Health Risk
## Score: .94926

In [7]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
DATA_DIR = Path("playground-series-s6e7")
TARGET = "health_condition"
ID_COL = "id"

rng = np.random.default_rng(SEED)
print("LightGBM", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

LightGBM 4.6.0 | pandas 3.0.3 | numpy 2.4.6


In [8]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train.shape, "| test:", test.shape)

CAT_COLS = ["diet_type", "stress_level", "sleep_quality",
            "physical_activity_level", "smoking_alcohol", "gender"]
NUM_COLS = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
            "step_count", "exercise_duration", "water_intake"]

CLASSES = ["at-risk", "unhealthy", "fit"]
class_to_int = {c: i for i, c in enumerate(CLASSES)}
int_to_class = {i: c for c, i in class_to_int.items()}
y = train[TARGET].map(class_to_int).to_numpy()

print("\nTarget distribution:")
print(train[TARGET].value_counts(normalize=True).round(4))

train: (690088, 15) | test: (295753, 14)

Target distribution:
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64


## Feature preparation

In [9]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df.copy()

    X["n_missing"] = df[NUM_COLS + CAT_COLS].isna().sum(axis=1)

    X["steps_per_cal"] = X["step_count"] / X["calorie_expenditure"]
    X["cal_per_min_exercise"] = X["calorie_expenditure"] / (X["exercise_duration"] + 1e-3)
    X["steps_per_min_exercise"] = X["step_count"] / (X["exercise_duration"] + 1e-3)

    for c in CAT_COLS:
        X[c] = X[c].fillna("__NA__").astype("category")

    return X


FEATURES = NUM_COLS + CAT_COLS + [
    "n_missing", "steps_per_cal", "cal_per_min_exercise", "steps_per_min_exercise",
]

X = build_features(train)[FEATURES]
X_test = build_features(test)[FEATURES]

for c in CAT_COLS:
    levels = X[c].cat.categories.union(X_test[c].cat.categories)
    X[c] = X[c].cat.set_categories(levels)
    X_test[c] = X_test[c].cat.set_categories(levels)

print("Features:", len(FEATURES))
print(X.dtypes)

Features: 17
sleep_duration              float64
heart_rate                  float64
bmi                         float64
calorie_expenditure         float64
step_count                  float64
exercise_duration           float64
water_intake                float64
diet_type                  category
stress_level               category
sleep_quality              category
physical_activity_level    category
smoking_alcohol            category
gender                     category
n_missing                     int64
steps_per_cal               float64
cal_per_min_exercise        float64
steps_per_min_exercise      float64
dtype: object


## Cross-validated training

In [10]:
lgb_params = dict(
    objective="binary",
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)


def fit_binary(Xtr, ytr, Xva, yva):
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(
        Xtr, ytr,
        eval_set=[(Xva, yva)],
        eval_metric="binary_logloss",
        categorical_feature=CAT_COLS,
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )
    return m


skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_proba = np.zeros((len(X), len(CLASSES)))
test_proba = np.zeros((len(X_test), len(CLASSES)))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
    ytr, yva = y[tr_idx], y[va_idx]

    # Stage 1: at-risk (0) vs not-at-risk (1) on all rows
    m1 = fit_binary(Xtr, (ytr != 0).astype(int), Xva, (yva != 0).astype(int))
    p_not_va = m1.predict_proba(Xva)[:, 1]
    p_not_test = m1.predict_proba(X_test)[:, 1]

    # Stage 2: unhealthy (1) vs fit (0) among non-at-risk rows only
    tr2 = ytr != 0
    va2 = yva != 0
    m2 = fit_binary(Xtr[tr2], (ytr[tr2] == 1).astype(int),
                    Xva[va2], (yva[va2] == 1).astype(int))
    p_unh_va = m2.predict_proba(Xva)[:, 1]
    p_unh_test = m2.predict_proba(X_test)[:, 1]

    oof_proba[va_idx, 0] = 1 - p_not_va
    oof_proba[va_idx, 1] = p_not_va * p_unh_va
    oof_proba[va_idx, 2] = p_not_va * (1 - p_unh_va)

    test_proba[:, 0] += (1 - p_not_test) / N_SPLITS
    test_proba[:, 1] += (p_not_test * p_unh_test) / N_SPLITS
    test_proba[:, 2] += (p_not_test * (1 - p_unh_test)) / N_SPLITS

    fold_bal = balanced_accuracy_score(yva, oof_proba[va_idx].argmax(1))
    print(f"fold {fold}: argmax bal-acc={fold_bal:.5f}")

argmax_oof = oof_proba.argmax(axis=1)
print(f"\nOOF balanced accuracy (plain argmax): {balanced_accuracy_score(y, argmax_oof):.5f}")

Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's binary_logloss: 0.111768
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[291]	valid_0's binary_logloss: 0.0378578
fold 0: argmax bal-acc=0.94425
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's binary_logloss: 0.112223
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[305]	valid_0's binary_logloss: 0.0363532
fold 1: argmax bal-acc=0.94536
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's binary_logloss: 0.114382
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[278]	valid_0's binary_logloss: 0.0420432
fold 2: argmax bal-acc=0.94407
Training until validation scores d

## Tune the decision rule for balanced accuracy

In [11]:
from scipy.optimize import minimize
from sklearn.isotonic import IsotonicRegression


def weighted_predict(proba: np.ndarray, w: np.ndarray) -> np.ndarray:
    return (proba * w).argmax(axis=1)


def _neg_bal(w2, proba, y_true):
    w = np.array([1.0, w2[0], w2[1]])
    return -balanced_accuracy_score(y_true, weighted_predict(proba, w))


def optimize_weights(proba, y_true, grid=None):
    grid = np.geomspace(0.3, 30, 30) if grid is None else grid
    best_w = np.array([1.0, 1.0, 1.0])
    best = -_neg_bal([1.0, 1.0], proba, y_true)
    for wu in grid:
        for wf in grid:
            s = -_neg_bal([wu, wf], proba, y_true)
            if s > best:
                best, best_w = s, np.array([1.0, wu, wf])
    res = minimize(_neg_bal, x0=best_w[1:], args=(proba, y_true),
                   method="Nelder-Mead", options=dict(xatol=1e-3, fatol=1e-7, maxiter=1000))
    if -res.fun > best:
        best, best_w = -res.fun, np.array([1.0, res.x[0], res.x[1]])
    return best_w, best


def calibrate(oof_p, test_p, y_true):
    oc, tc = np.zeros_like(oof_p), np.zeros_like(test_p)
    for c in range(oof_p.shape[1]):
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(oof_p[:, c], (y_true == c).astype(float))
        oc[:, c] = iso.transform(oof_p[:, c])
        tc[:, c] = iso.transform(test_p[:, c])
    oc /= oc.sum(axis=1, keepdims=True)
    tc /= tc.sum(axis=1, keepdims=True)
    return oc, tc


w_raw, s_raw = optimize_weights(oof_proba, y)
oof_cal, test_cal = calibrate(oof_proba, test_proba, y)
w_cal, s_cal = optimize_weights(oof_cal, y)

print(f"argmax (no tuning): {balanced_accuracy_score(y, argmax_oof):.5f}")
print(f"raw blend        : {s_raw:.5f}  w={w_raw.round(3)}")
print(f"calibrated blend : {s_cal:.5f}  w={w_cal.round(3)}")

if s_cal >= s_raw:
    oof_final, test_final, best_w, best_score = oof_cal, test_cal, w_cal, s_cal
    print("-> using CALIBRATED probabilities")
else:
    oof_final, test_final, best_w, best_score = oof_proba, test_proba, w_raw, s_raw
    print("-> using RAW probabilities")

tuned_oof = weighted_predict(oof_final, best_w)
print(f"\nFinal OOF balanced accuracy (tuned): {best_score:.5f}")
print("\nConfusion matrix (rows=true, cols=pred) [at-risk, unhealthy, fit]:")
print(confusion_matrix(y, tuned_oof))
print("\n", classification_report(y, tuned_oof, target_names=CLASSES, digits=4))

argmax (no tuning): 0.94336
raw blend        : 0.94959  w=[1.    2.723 4.42 ]
calibrated blend : 0.94962  w=[ 1.     9.748 22.383]
-> using CALIBRATED probabilities

Final OOF balanced accuracy (tuned): 0.94962

Confusion matrix (rows=true, cols=pred) [at-risk, unhealthy, fit]:
[[553481  24666  14414]
 [  1786  55657    281]
 [  1765    200  37838]]

               precision    recall  f1-score   support

     at-risk     0.9936    0.9340    0.9629    592561
   unhealthy     0.6912    0.9642    0.8052     57724
         fit     0.7203    0.9506    0.8196     39803

    accuracy                         0.9375    690088
   macro avg     0.8017    0.9496    0.8626    690088
weighted avg     0.9526    0.9375    0.9415    690088



## Predict test set & write submission

In [12]:
test_pred_int = weighted_predict(test_final, best_w)
test_pred_lbl = pd.Series(test_pred_int).map(int_to_class)

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_pred_lbl})
submission.to_csv("submission.csv", index=False)

print("Wrote submission.csv:", submission.shape)
print("\nPredicted label distribution:")
print(submission[TARGET].value_counts(normalize=True).round(4))
print("\nTrain label distribution (for reference):")
print(train[TARGET].value_counts(normalize=True).round(4))
submission.head()

Wrote submission.csv: (295753, 2)

Predicted label distribution:
health_condition
at-risk      0.8086
unhealthy    0.1165
fit          0.0749
Name: proportion, dtype: float64

Train label distribution (for reference):
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
